# Nekomimi Waifu Seeker — Colab (L4) + Ollama

**What each model does**

- **Laya** (torch, downloaded from Hugging Face as `convaiinnovations/laya`) is the decision engine. It chooses questions, scores candidates, and decides when to guess. It never generates text, and it does not call Ollama.
- **Ollama** runs the optional query rewriter in `waifu_engine/query_llm.py`. With `WAIFU_QUERY_LLM=1` it turns text you typed into web-search phrases. It never writes questions and never guesses. Template queries are used whenever it is off or a call fails.

**Colab secrets** (key icon in the left sidebar). Cells read them with `google.colab.userdata.get`. No token is written into this notebook.

| Secret | Required | Purpose |
|---|---|---|
| `NGROK_TOKEN` | yes | ngrok authtoken; the public URL is printed after the server starts |
| `HF_TOKEN` | no | Hugging Face token so the Laya weight download is authenticated |
| `GOOGLE_API_KEY` | no | Gemini search grounding. Search stays off when this secret is missing |

**Run order.** Runtime → Change runtime type → **L4**, then run every cell from the top.

1. Config (`REPO_URL`, `BRANCH=main`, Ollama model).
2. Load secrets into the environment.
3. Clone this repo and check out `main`.
4. Install Ollama and Python deps, install Node 22 if needed, then `cd webui && npm ci && npm run build`.
5. Start Ollama, pull the GGUF, and warm it up.
6. Patch the cloned `query_llm.py` for Ollama, then smoke-test the rewriter.
7. Start uvicorn on `APP_PORT` and open the printed `/nekomimi` link.

The browser UI is the SolidJS bundle `waifu_engine/webui_dist`. FastAPI answers `/` and `/nekomimi` with **503** until `npm run build` has written that directory. Docker does the same build (`npm ci` then `npm run build` on Node 22). A local checkout can use `npm install` instead of `npm ci`; this notebook follows the lockfile, matching the image.


In [ ]:
# 1. Config
REPO_URL   = "https://github.com/CooLguNxDD/Nekomimi-Waifu-Seeker.git"
BRANCH     = "main"
APP_DIR    = "Nekomimi-Waifu-Seeker"

# Query rewriter only. Laya is a separate Hugging Face download.
HF_REPO    = "unsloth/Qwen3.6-35B-A3B-GGUF"
QUANT      = "UD-Q3_K_M"     # ~17 GB
CTX        = 10000
MODEL_NAME = "Qwen3.6-35B-A3B-GGUF"

THINKING   = False            # False = no reasoning: faster, and JSON lands in the reply
OLLAMA_URL = "http://127.0.0.1:11434"
APP_PORT   = 7860


In [ ]:
import os
from google.colab import userdata

# 2. Secrets and Gemini switches. Missing optional secrets leave that feature off.
os.environ["USE_TF"] = "0"  # Transformers can hang while probing TensorFlow

try:
    os.environ["NGROK_TOKEN"] = userdata.get("NGROK_TOKEN")
except Exception:
    print("Warning: NGROK_TOKEN secret not found. The public URL cell will fail until you add it.")

try:
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except Exception:
    print("Warning: GOOGLE_API_KEY secret not found. Gemini search stays off.")

try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    print("(no HF_TOKEN secret — Laya download will be unauthenticated, that's fine)")

os.environ["WAIFU_GEMINI_MODEL"] = "gemini-3.8-flash"
os.environ["WAIFU_GEMINI_SEARCH"] = "1"
os.environ["WAIFU_GEMINI_LLM"] = "0"   # Ollama is the query rewriter; Gemini is search only
os.environ["WAIFU_GEMINI_TIMEOUT"] = "15"

print("Secrets and Gemini settings loaded.")
print("NGROK_TOKEN set:", bool(os.environ.get("NGROK_TOKEN")))
print("GOOGLE_API_KEY set:", bool(os.environ.get("GOOGLE_API_KEY")))
print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))


In [ ]:
# 3. Clone repo + checkout branch (idempotent)
import os, subprocess
if not os.path.isdir(APP_DIR):
    subprocess.run(["git", "clone", REPO_URL, APP_DIR], check=True)
subprocess.run(["git", "fetch", "--all", "--prune", "-q"], cwd=APP_DIR, check=True)
subprocess.run(["git", "checkout", "-B", BRANCH], cwd=APP_DIR, check=True)
subprocess.run(["git", "reset", "--hard", f"origin/{BRANCH}"], cwd=APP_DIR, check=True)
print(subprocess.run(["git", "log", "-1", "--oneline"], cwd=APP_DIR, capture_output=True, text=True).stdout)


In [ ]:
# 4. Ollama, Python deps, and the SolidJS UI.
# FastAPI serves waifu_engine/webui_dist and returns 503 until this build exists.
import os, subprocess, sys

os.environ["USE_TF"] = "0"

!sudo apt-get update -o Acquire::Retries=3 -qq --allow-releaseinfo-change
!sudo DEBIAN_FRONTEND=noninteractive apt-get install -y -qq zstd xz-utils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1

def _node_major():
    """Major version of `node` on PATH, or 0 when Node is missing."""
    try:
        out = subprocess.check_output(["node", "-p", "process.versions.node"], text=True).strip()
        return int(out.split(".")[0])
    except (OSError, subprocess.CalledProcessError, ValueError):
        return 0

if _node_major() < 22:
    # Vite 7 needs a current Node. The Docker UI stage uses node:22.
    script = "\n".join([
        "set -euo pipefail",
        "ver=$(curl -fsSL https://nodejs.org/dist/latest-v22.x/SHASUMS256.txt "
        "| awk '/node-v22\\..*-linux-x64\\.tar\\.xz$/{print $2; exit}')",
        'case "$ver" in',
        "  node-v22.*) ;;",
        '  *) echo "could not resolve a Node 22 tarball: ${ver}" >&2; exit 1 ;;',
        "esac",
        'curl -fsSL "https://nodejs.org/dist/latest-v22.x/${ver}" '
        "| sudo tar -xJ -C /usr/local --strip-components=1",
    ])
    subprocess.run(["bash", "-lc", script], check=True)
print("node", subprocess.check_output(["node", "--version"], text=True).strip())

!pip install -q pyngrok -r {APP_DIR}/requirements.txt
# playwright: headless Chromium search. gemini: Google Search grounding when GOOGLE_API_KEY is set.
!pip install -q -e "./{APP_DIR}[playwright,gemini]"
pw = subprocess.run([sys.executable, "-m", "playwright", "install", "chromium"])
if pw.returncode != 0:
    print("Playwright Chromium did not install. Search still uses Wikipedia, AniList, and DuckDuckGo.")

ui = os.path.join(APP_DIR, "webui")
subprocess.run(["npm", "ci"], cwd=ui, check=True)
subprocess.run(["npm", "run", "build"], cwd=ui, check=True)
dist = os.path.join(APP_DIR, "waifu_engine", "webui_dist", "index.html")
if not os.path.isfile(dist):
    raise RuntimeError(f"UI build did not write {dist}")
print("UI bundle:", dist)
!ollama --version


In [ ]:
# 5. Start Ollama server
# `pkill -x` matches the process name exactly. `pkill -f ollama` also matches
# the shell running that very command and can kill it.
import os, subprocess, time, urllib.request
subprocess.run(["pkill", "-x", "ollama"])
time.sleep(1)

ollama_env = os.environ.copy()
ollama_env["OLLAMA_FLASH_ATTENTION"] = "1"
ollama_env["OLLAMA_KV_CACHE_TYPE"]   = "q8_0"
ollama_env["OLLAMA_KEEP_ALIVE"]      = "-1"   # keep model loaded; no cold starts mid-game
ollama_log = open("ollama.log", "w")
ollama_proc = subprocess.Popen(["ollama", "serve"], stdout=ollama_log, stderr=ollama_log, env=ollama_env)

for _ in range(30):
    try:
        urllib.request.urlopen(OLLAMA_URL)
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Ollama didn't start — check ollama.log")
print("Ollama up, pid", ollama_proc.pid)


In [ ]:
# 6. Pull model + wrap with fixed context
from huggingface_hub import list_repo_files
assert any(QUANT in f for f in list_repo_files(HF_REPO) if f.endswith(".gguf")), f"{QUANT} not in {HF_REPO}"

!ollama pull hf.co/{HF_REPO}:{QUANT} 2>&1 | tail -n 1
with open("Modelfile", "w") as f:
    f.write(f"FROM hf.co/{HF_REPO}:{QUANT}\nPARAMETER num_ctx {CTX}\n")
!ollama create {MODEL_NAME} -f Modelfile 2>&1 | tail -n 1
!ollama list


In [ ]:
# 7. Warm up via the same OpenAI-compatible endpoint the app will use
import json, time, urllib.request

def ollama_chat(messages, **extra):
    """One chat completion against the local Ollama OpenAI endpoint."""
    body = {"model": MODEL_NAME, "messages": messages, "max_tokens": 2048, **extra}
    req = urllib.request.Request(
        f"{OLLAMA_URL}/v1/chat/completions",
        data=json.dumps(body).encode(),
        headers={"Content-Type": "application/json"},
    )
    t = time.time()
    with urllib.request.urlopen(req, timeout=180) as r:
        out = json.loads(r.read())
    return out, time.time() - t

extra = {} if THINKING else {"reasoning_effort": "none"}
out, dt = ollama_chat([{"role": "user", "content": "Introduce yourself in one sentence."}], **extra)
msg = out["choices"][0]["message"]
print(f"{dt:.1f}s | content: {msg.get('content')!r}")
print("thought anyway:", bool(msg.get("reasoning")), "(should be False when THINKING=False)")
u = out.get("usage") or {}
print("usage:", u)
if u.get("completion_tokens"):
    print(f"≈ {u['completion_tokens']/dt:.1f} tok/s (incl. prompt time)")
!ollama ps   # PROCESSOR must say "100% GPU". Any "% CPU" means layers spilled to RAM.
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv


### Patch `query_llm.py` for Ollama

`main` caps the rewriter at `max_tokens: 96` and turns Qwen thinking off with `chat_template_kwargs.enable_thinking`. vLLM and SGLang honour that switch. Ollama ignores it, so the model can spend the whole budget on reasoning, return no JSON, and the app quietly falls back to template queries.

This cell edits **only the Colab clone**. It adds two env vars the next cell sets:

- `WAIFU_QUERY_LLM_MAX_TOKENS` — token cap (the default stays 96)
- `WAIFU_QUERY_LLM_THINKING=0` — also send `reasoning_effort: "none"`, the switch Ollama honours

Leaving `WAIFU_QUERY_LLM_THINKING` unset keeps the upstream body, so a vLLM or OpenAI endpoint is unchanged. Re-running the cell is safe. `THINKING` in the config cell is the switch. The cell stops if those anchors have moved upstream, instead of skipping a patch that would no longer apply.


In [ ]:
# 8. Patch the cloned query_llm.py for Ollama.
#    a) max_tokens from WAIFU_QUERY_LLM_MAX_TOKENS (upstream literal is 96)
#    b) WAIFU_QUERY_LLM_THINKING=0 -> reasoning_effort="none"
p = f"{APP_DIR}/waifu_engine/query_llm.py"
src = open(p).read()

old_a = '"max_tokens": 96,'
new_a = '"max_tokens": int(os.getenv("WAIFU_QUERY_LLM_MAX_TOKENS", "96")),'

old_b = "    return body\n\n\ndef parse_queries"
new_b = (
    "    thinking = os.getenv(\"WAIFU_QUERY_LLM_THINKING\")\n"
    "    if thinking is not None and thinking.lower() not in _YES and not _is_openai(base_url()):\n"
    "        # Ollama ignores chat_template_kwargs; reasoning_effort=\"none\" disables thinking there.\n"
    "        body[\"reasoning_effort\"] = \"none\"\n"
    "    return body\n\n\ndef parse_queries"
)

for name, old, new in [("max_tokens", old_a, new_a), ("thinking flag", old_b, new_b)]:
    if new in src:
        print(f"{name}: already patched")
    elif old in src:
        src = src.replace(old, new, 1)
        print(f"{name}: patched")
    else:
        raise RuntimeError(
            f"{name}: anchor not found in {p}. "
            "query_llm.py changed on main; update this cell before relying on Ollama."
        )
open(p, "w").write(src)


In [ ]:
# 9. App env — names query_llm.py and the Gemini source actually read.
#    HF_TOKEN was loaded with the other secrets. The dummy API key is only a
#    bearer for the local Ollama server; it is not an account credential.
app_env = os.environ.copy()
app_env.update({
    "USE_TF": "0",
    "WAIFU_QUERY_LLM":            "1",
    "WAIFU_QUERY_LLM_BASE_URL":   f"{OLLAMA_URL}/v1",
    "WAIFU_QUERY_LLM_MODEL":      MODEL_NAME,
    "WAIFU_QUERY_LLM_API_KEY":    "ollama",
    "WAIFU_QUERY_LLM_TIMEOUT":    "90",
    "WAIFU_QUERY_LLM_THINKING":   "1" if THINKING else "0",
    # thinking off: the answer is a short JSON array, so a small cap bounds a runaway reply
    "WAIFU_QUERY_LLM_MAX_TOKENS": "2048" if THINKING else "256",
})
shown = {k: v for k, v in app_env.items() if k.startswith("WAIFU_")}
print(shown)


In [ ]:
# 10. Smoke-test the rewriter with the same env as the server.
#     _call raises on transport errors. rewrite() swallows them, which hides a dead Ollama.
smoke = """import time, traceback
from waifu_engine import query_llm as q
print("enabled:", q.enabled(), "| url:", q.base_url(), "| model:", q.model())
facts = ("cat ears", "silver hair", "from a video game")
t = time.time()
try:
    print("queries:", q._call(facts, None, 3, q.base_url(), q.model()))
except Exception:
    traceback.print_exc()
print(f"took {time.time()-t:.1f}s")
"""
path = "/tmp/colab_query_smoke.py"
open(path, "w").write(smoke)
r = subprocess.run(
    ["python", path], cwd=APP_DIR, env=app_env,
    capture_output=True, text=True, timeout=300,
)
print(r.stdout, r.stderr[-3000:])
# Expect a tuple of search strings.
#   None    -> reply had no JSON array (thinking ate the budget; raise MAX_TOKENS or set THINKING False)
#   timeout -> raise WAIFU_QUERY_LLM_TIMEOUT


In [ ]:
# 11. Start the web server, wait until it is listening, then open ngrok.
#     The UI bundle must already exist or every page returns 503.
import socket, time
from pyngrok import ngrok

dist = os.path.join(APP_DIR, "waifu_engine", "webui_dist", "index.html")
if not os.path.isfile(dist):
    raise RuntimeError("web UI is not built. Re-run the install cell (npm ci && npm run build).")

try:
    server.terminate()
    server.wait(timeout=15)
except Exception:
    pass

server_log = open("server.log", "w")
# web.main() always binds 7860. Pass APP_PORT here so the wait loop and ngrok
# watch the same port. Host stays 127.0.0.1: uvicorn is IPv4-only, and
# "localhost" can make ngrok dial ::1.
server = subprocess.Popen(
    ["python", "-m", "uvicorn", "waifu_engine.web:app",
     "--host", "127.0.0.1", "--port", str(APP_PORT)],
    cwd=APP_DIR, stdout=server_log, stderr=server_log, env=app_env,
)

def port_open(port):
    """True when something on this machine accepts TCP connections on `port`."""
    with socket.socket() as s:
        s.settimeout(1)
        return s.connect_ex(("127.0.0.1", port)) == 0

# First start also downloads Laya (~800 MB) before the port opens.
for i in range(600):
    if server.poll() is not None:
        print(open("server.log").read())
        raise RuntimeError("server exited")
    if port_open(APP_PORT):
        break
    time.sleep(1)
else:
    print(open("server.log").read()[-4000:])
    raise RuntimeError("server never opened the port — see server.log")
print(f"server listening after ~{i}s")

if not os.environ.get("NGROK_TOKEN"):
    raise RuntimeError("NGROK_TOKEN is empty. Add it under Colab secrets and re-run the secrets cell.")

ngrok.kill()
ngrok.set_auth_token(os.environ["NGROK_TOKEN"])
# 127.0.0.1 explicitly: uvicorn binds IPv4 only; "localhost" can make ngrok dial ::1.
tunnel = ngrok.connect(f"127.0.0.1:{APP_PORT}")
print(
    "Warning: this ngrok URL is public and has no login. "
    "Anyone with the link can play. If GOOGLE_API_KEY is set, "
    "their games can trigger Gemini Search calls billed to that key."
)
print("public URL:", tunnel.public_url + "/nekomimi")


In [ ]:
# 12. After a few rounds: server log tail, and whether the app called Ollama.
print("--- server.log (tail) ---")
print("".join(open("server.log").readlines()[-20:]))
print("--- Ollama chat requests seen ---")
hits = [line for line in open("ollama.log") if "/v1/chat/completions" in line]
print(f"{len(hits)} request(s)")
print("".join(hits[-5:]))


In [ ]:
# 13. (optional) Live-tail server.log — interrupt to stop
with open("server.log") as f:
    f.seek(0, os.SEEK_END)
    try:
        while True:
            line = f.readline()
            if line:
                print(line, end="", flush=True)
            else:
                time.sleep(0.5)
    except KeyboardInterrupt:
        print("\nstopped")
